# Gold 3_17 — Communication Recency

Direction-aware communication recency per client, plus a `whose_turn` classification. One row per client in the `customer_health_profile` population.

**This is a pure context table.** It is NOT an input to the health score, and it changes no existing table, metric, or business rule. Applying it (e.g. "who do we contact today") is downstream presentation logic the business owns.

## Contact vs attempt — the core distinction

**Contact made** (a message was actually delivered in that direction):
* Email in (`status = 'inbox'`) — client wrote to us
* Email out (`status = 'sent'`) — we delivered a message
* Call in answered — two-way conversation
* Call in unanswered / voicemail — the client tried to reach us (their move)
* Call out `answered` or `vm-left` (a delivered voicemail) — we reached them
* Text (SMS) in / out — delivered either way (source has no delivery disposition)
* Slack student message (in) / team message (out) — via `gold.sentiment`
* Meeting — mutual, counts as contact in BOTH directions

**Attempt only** (no delivery — never flips whose turn it is):
* Call out `no-answer` / `busy` / `error` / `blocked` / null disposition
* Email `draft` / `outbox` / `scheduled`

## Whose turn

Compare the most recent **contact-made** event on each side; attempts are ignored:
* `OUR TURN` — client's latest inbound contact is newer than our latest delivered message (they're waiting on us)
* `THEIR TURN` — our latest delivered message is newer than their latest inbound contact
* `MUTUAL` — both sides' latest contact is the same event/day (meeting, same-day exchange)
* `NEVER REACHED` — outbound attempts exist, but no contact was ever established
* `NO CONTACT` — no communication of any kind recorded

Example: client emailed 2 days ago, we called out yesterday with no answer — the call is an attempt, so the last contact is still their email — **OUR TURN** (the email still needs answering).

## Provisional constants (NOT health-score rules — business may retune)

* Outreach exhaustion: at least 3 distinct channels with outbound activity in the last 14 days AND no inbound contact in the last 30 days
* Note: unlike the engagement email metric, the Calendly scheduling-notification exclusion is deliberately NOT applied here — this table counts raw communication, not engagement.

In [0]:
%sql
-- COMMUNICATION_RECENCY — one row per client: per-channel inbound/outbound recency,
-- contact-vs-attempt split, cross-channel summary, whose_turn, outreach-exhaustion flag.
-- CONTEXT ONLY — not an input to the health score.
CREATE OR REPLACE TABLE crm_ingestion.gold.communication_recency AS (
WITH act AS (
    SELECT
        f.lead_id,
        -- EMAIL (Close status encodes direction: inbox = incoming, sent = outgoing)
        MAX(CASE WHEN f.type = 'Email' AND f.status = 'inbox' THEN f.activity_at END) AS email_in_at,
        MAX(CASE WHEN f.type = 'Email' AND f.status = 'sent' THEN f.activity_at END) AS email_out_contact_at,
        MAX(CASE WHEN f.type = 'Email' AND f.status IN ('draft', 'outbox', 'scheduled') THEN f.activity_at END) AS email_out_attempt_at,
        -- CALL (direction + disposition)
        MAX(CASE WHEN f.type = 'Call' AND f.direction = 'inbound' AND f.disposition = 'answered' THEN f.activity_at END) AS call_in_answered_at,
        MAX(CASE WHEN f.type = 'Call' AND f.direction = 'inbound' AND (f.disposition IS NULL OR f.disposition != 'answered') THEN f.activity_at END) AS call_in_unanswered_at,
        MAX(CASE WHEN f.type = 'Call' AND f.direction = 'outbound' AND f.disposition IN ('answered', 'vm-left') THEN f.activity_at END) AS call_out_contact_at,
        MAX(CASE WHEN f.type = 'Call' AND f.direction = 'outbound' AND (f.disposition IS NULL OR f.disposition NOT IN ('answered', 'vm-left')) THEN f.activity_at END) AS call_out_attempt_at,
        -- TEXT / SMS
        MAX(CASE WHEN f.type = 'SMS' AND f.direction = 'inbound' THEN f.activity_at END) AS text_in_at,
        MAX(CASE WHEN f.type = 'SMS' AND f.direction = 'outbound' THEN f.activity_at END) AS text_out_at,
        -- MEETING (mutual) and other activity (Note, CustomActivity, Created, LeadMerge)
        MAX(CASE WHEN f.type = 'Meeting' THEN f.activity_at END) AS meeting_at,
        MAX(CASE WHEN f.type NOT IN ('Email', 'Call', 'SMS', 'Meeting') THEN f.activity_at END) AS other_activity_at,
        MAX(f.activity_at) AS any_activity_at
    FROM crm_ingestion.gold.leads_activity_fact f
    WHERE f.type IS NOT NULL
    GROUP BY f.lead_id
),
days AS (
    SELECT
        a.lead_id,
        DATEDIFF(current_date, TO_DATE(a.email_in_at))            AS email_in_days,
        DATEDIFF(current_date, TO_DATE(a.email_out_contact_at))   AS email_out_days,
        DATEDIFF(current_date, TO_DATE(a.email_out_attempt_at))   AS email_attempt_days,
        DATEDIFF(current_date, TO_DATE(a.call_in_answered_at))    AS call_in_answered_days,
        DATEDIFF(current_date, TO_DATE(a.call_in_unanswered_at))   AS call_in_unanswered_days,
        DATEDIFF(current_date, TO_DATE(a.call_out_contact_at))    AS call_out_contact_days,
        DATEDIFF(current_date, TO_DATE(a.call_out_attempt_at))    AS call_out_attempt_days,
        DATEDIFF(current_date, TO_DATE(a.text_in_at))              AS text_in_days,
        DATEDIFF(current_date, TO_DATE(a.text_out_at))             AS text_out_days,
        DATEDIFF(current_date, TO_DATE(a.meeting_at))             AS meeting_days,
        DATEDIFF(current_date, TO_DATE(a.other_activity_at))      AS other_activity_days,
        DATEDIFF(current_date, TO_DATE(a.any_activity_at))        AS any_activity_days
    FROM act a
),
base AS (
    SELECT
        p.lead_id,
        d.email_in_days, d.email_out_days, d.email_attempt_days,
        d.call_in_answered_days, d.call_in_unanswered_days,
        d.call_out_contact_days, d.call_out_attempt_days,
        d.text_in_days, d.text_out_days,
        d.meeting_days, d.other_activity_days, d.any_activity_days,
        s.days_since_last_message_from_client      AS slack_in_days,
        s.days_since_last_message_from_team_member AS slack_out_days
    FROM (SELECT lead_id FROM crm_ingestion.gold.customer_health_profile
          WHERE VRSN_END_DATE IS NULL) p  -- SCD2: open versions = current clients
    LEFT JOIN days d                         ON d.lead_id = p.lead_id
    LEFT JOIN crm_ingestion.gold.sentiment s ON s.lead_id = p.lead_id
),
cross_channel AS (
    SELECT
        b.*,
        -- Most recent contact-made on each side (attempts excluded; meeting counts both ways)
        LEAST(b.email_in_days, b.call_in_answered_days, b.call_in_unanswered_days,
              b.text_in_days, b.slack_in_days, b.meeting_days)       AS last_contact_in_days,
        LEAST(b.email_out_days, b.call_out_contact_days, b.text_out_days,
              b.slack_out_days, b.meeting_days)                       AS last_contact_out_days,
        LEAST(b.email_attempt_days, b.call_out_attempt_days)          AS last_attempt_days,
        -- Distinct channels with outbound activity (contact OR attempt) in the last 14 days
        (CASE WHEN b.email_out_days <= 14 OR b.email_attempt_days <= 14 THEN 1 ELSE 0 END
       + CASE WHEN b.call_out_contact_days <= 14 OR b.call_out_attempt_days <= 14 THEN 1 ELSE 0 END
       + CASE WHEN b.text_out_days <= 14 THEN 1 ELSE 0 END
       + CASE WHEN b.slack_out_days <= 14 THEN 1 ELSE 0 END)          AS channels_out_14d
    FROM base b
)
SELECT
    lead_id,
    -- per-channel inbound
    email_in_days, call_in_answered_days, call_in_unanswered_days, text_in_days, slack_in_days,
    -- per-channel outbound: contact vs attempt
    email_out_days, email_attempt_days,
    call_out_contact_days, call_out_attempt_days,
    text_out_days, slack_out_days,
    -- mutual / other
    meeting_days, other_activity_days, any_activity_days,
    -- cross-channel summary
    last_contact_in_days, last_contact_out_days, last_attempt_days, channels_out_14d,
    -- whose turn (attempts never flip the turn — contact-made only)
    CASE
        WHEN last_contact_in_days IS NULL AND last_contact_out_days IS NULL
             AND any_activity_days IS NULL AND slack_in_days IS NULL AND slack_out_days IS NULL
            THEN 'NO CONTACT'
        WHEN last_contact_in_days IS NULL AND last_contact_out_days IS NULL
            THEN 'NEVER REACHED'
        WHEN last_contact_in_days = last_contact_out_days
            THEN 'MUTUAL'
        WHEN last_contact_out_days IS NULL OR last_contact_in_days < last_contact_out_days
            THEN 'OUR TURN'
        ELSE 'THEIR TURN'
    END AS whose_turn,
    -- PROVISIONAL: outreach exhaustion (>=3 channels outbound in 14d, no inbound contact in 30d)
    CASE WHEN channels_out_14d >= 3
          AND (last_contact_in_days IS NULL OR last_contact_in_days > 30)
        THEN 1 ELSE 0 END AS exhausted_outreach_flag
FROM cross_channel
)

In [0]:
%sql
-- Validation: population must equal customer_health_profile (one row per client)
SELECT
    whose_turn,
    exhausted_outreach_flag,
    COUNT(*) AS clients
FROM crm_ingestion.gold.communication_recency
GROUP BY whose_turn, exhausted_outreach_flag
ORDER BY clients DESC

In [0]:
%sql
-- Validation: HOW STALE is each turn? (SLA lens: premium coaching — 24h
-- response expected. 0-1 days = inside SLA; past 3 days = MAJOR; past a
-- week = unacceptable. A count is not an SLA read.)
-- Age of the move that created the turn:
--   OUR TURN  = days since THEIR last inbound contact  (age of the debt we owe)
--   THEIR TURN = days since OUR last delivered contact (freshness of our touch)
--   MUTUAL    = same-day event, age = days since it
--   NO CONTACT / NEVER REACHED = no contact-made event to age
-- Non-client leads whose_turn='US' (reached out, we never connected back)
-- are shown on the same scale — these are inbound interest going unanswered.
WITH client_turns AS (
    SELECT
        'client' AS population,
        cr.whose_turn,
        CASE WHEN cr.whose_turn IN ('NO CONTACT', 'NEVER REACHED') THEN NULL
             WHEN cr.whose_turn = 'OUR TURN'                        THEN cr.last_contact_in_days
             ELSE cr.last_contact_out_days
        END AS turn_age_days
    FROM crm_ingestion.gold.communication_recency cr
),
nonclient_turns AS (
    SELECT
        'non-client lead' AS population,
        'OUR TURN'        AS whose_turn,
        cs.client_reach_days AS turn_age_days
    FROM crm_ingestion.gold.communication_state cs
    LEFT JOIN crm_ingestion.gold.customer_health_profile p ON p.lead_id = cs.lead_id
    WHERE p.lead_id IS NULL
      AND cs.whose_turn = 'US'
),
all_turns AS (
    SELECT population, whose_turn, turn_age_days FROM client_turns
    UNION ALL
    SELECT population, whose_turn, turn_age_days FROM nonclient_turns
),
bucketed AS (
    SELECT
        population, whose_turn, turn_age_days,
        CASE
            WHEN turn_age_days IS NULL THEN 'no contact ever'
            WHEN turn_age_days <= 1  THEN '0-1 days (inside 24h SLA)'
            WHEN turn_age_days <= 2  THEN '2 days'
            WHEN turn_age_days <= 3  THEN '3 days'
            WHEN turn_age_days <= 7  THEN '4-7 days (major)'
            WHEN turn_age_days <= 14 THEN '8-14 days (>1 wk)'
            WHEN turn_age_days <= 30 THEN '15-30 days (>2 wks)'
            ELSE '31+ days (stale)'
        END AS age_bucket,
        CASE
            WHEN turn_age_days IS NULL THEN 0
            WHEN turn_age_days <= 1  THEN 1
            WHEN turn_age_days <= 2  THEN 2
            WHEN turn_age_days <= 3  THEN 3
            WHEN turn_age_days <= 7  THEN 4
            WHEN turn_age_days <= 14 THEN 5
            WHEN turn_age_days <= 30 THEN 6
            ELSE 7
        END AS age_rank
    FROM all_turns
)
SELECT population, whose_turn, age_bucket, COUNT(*) AS leads
FROM bucketed
GROUP BY population, whose_turn, age_bucket, age_rank
ORDER BY population, whose_turn, age_rank

In [0]:
%sql
-- Validation 2: HOW STALE is "OUR TURN"? (SLA lens: premium coaching — a
-- paying client should hear back within 24 hours. 0-1 days = inside SLA;
-- 2-3 days = slipping; past 3 days = MAJOR; past a week = unacceptable.)
WITH our_turn AS (
    SELECT
        CASE WHEN last_contact_in_days <= 1  THEN 0
             WHEN last_contact_in_days <= 2  THEN 1
             WHEN last_contact_in_days <= 3  THEN 2
             WHEN last_contact_in_days <= 7  THEN 3
             WHEN last_contact_in_days <= 14 THEN 4
             WHEN last_contact_in_days <= 30 THEN 5
             ELSE 6 END AS ord,
        CASE WHEN last_contact_in_days <= 1  THEN '0-1 days (inside 24h SLA)'
             WHEN last_contact_in_days <= 2  THEN '2 days (slipping)'
             WHEN last_contact_in_days <= 3  THEN '3 days (last acceptable day)'
             WHEN last_contact_in_days <= 7  THEN '4-7 days (MAJOR)'
             WHEN last_contact_in_days <= 14 THEN '8-14 days (unacceptable)'
             WHEN last_contact_in_days <= 30 THEN '15-30 days (unacceptable)'
             ELSE '> 30 days (dropped conversation)' END AS waiting_since_bucket
    FROM crm_ingestion.gold.communication_recency
    WHERE whose_turn = 'OUR TURN'
)
SELECT
    waiting_since_bucket,
    COUNT(*) AS clients_waiting_on_us,
    ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_our_turn
FROM our_turn
GROUP BY ord, waiting_since_bucket
ORDER BY ord

In [0]:
%sql
-- Validation: DID WE EVEN TRY? Same 24h-SLA buckets, split by whether ANY
-- outbound attempt (unanswered call included) happened after their last
-- inbound. Separates ops friction ("we called, no answer") from negligence
-- ("nobody touched this lead"). Covers clients AND non-client leads —
-- the same standard applied to both populations.
WITH turns AS (
    SELECT
        'client' AS population,
        cr.last_contact_in_days AS waiting_days,
        CASE WHEN cr.last_attempt_days IS NOT NULL
              AND cr.last_attempt_days < cr.last_contact_in_days
             THEN 1 ELSE 0 END AS tried_since_reach
    FROM crm_ingestion.gold.communication_recency cr
    WHERE cr.whose_turn = 'OUR TURN'
    UNION ALL
    SELECT
        'non-client lead',
        cs.client_reach_days,
        CASE WHEN cs.we_attempted_days IS NOT NULL
              AND cs.we_attempted_days < cs.client_reach_days
             THEN 1 ELSE 0 END
    FROM crm_ingestion.gold.communication_state cs
    LEFT JOIN crm_ingestion.gold.customer_health_profile p
           ON p.lead_id = cs.lead_id AND p.VRSN_END_DATE IS NULL
    WHERE p.lead_id IS NULL
      AND cs.whose_turn = 'US'
),
buckets AS (
    SELECT
        population, tried_since_reach, waiting_days,
        CASE WHEN waiting_days <= 1  THEN 0
             WHEN waiting_days =  2  THEN 1
             WHEN waiting_days =  3  THEN 2
             WHEN waiting_days <= 7  THEN 3
             WHEN waiting_days <= 14 THEN 4
             WHEN waiting_days <= 30 THEN 5
             ELSE 6 END AS ord,
        CASE WHEN waiting_days <= 1  THEN '0-1 days (inside 24h SLA)'
             WHEN waiting_days =  2  THEN '2 days (slipping)'
             WHEN waiting_days =  3  THEN '3 days (last acceptable day)'
             WHEN waiting_days <= 7  THEN '4-7 days (MAJOR)'
             WHEN waiting_days <= 14 THEN '8-14 days (unacceptable)'
             WHEN waiting_days <= 30 THEN '15-30 days (unacceptable)'
             ELSE '> 30 days (dropped)' END AS waiting_since_bucket
    FROM turns
)
SELECT
    population,
    waiting_since_bucket,
    COUNT(*) AS waiting,
    SUM(tried_since_reach) AS we_tried_since,
    SUM(1 - tried_since_reach) AS we_did_nothing,
    ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY population), 1) AS pct
FROM buckets
GROUP BY population, ord, waiting_since_bucket
ORDER BY population, ord

In [0]:
%sql
-- Validation: does unanswered silence already show up in the health score?
-- Business hypothesis (from the client's chair): a paying coaching client left
-- waiting is a client heading for the exit — "the churn rate should be much
-- higher" than the score suggests. Test: OUR TURN staleness vs health_band.
-- Honest caveat: partially circular — engagement metrics feed the score — but
-- the score is multi-channel (logins, meetings, sentiment, sales), so pure
-- email silence alone should NOT collapse a band.
WITH waiting AS (
    SELECT
        cr.last_contact_in_days,
        p.HEALTH_BAND,
        p.FINAL_SCORE AS final_health_score
    FROM crm_ingestion.gold.communication_recency cr
    JOIN crm_ingestion.gold.customer_health_profile p
          ON p.lead_id = cr.lead_id AND p.VRSN_END_DATE IS NULL
    WHERE cr.whose_turn = 'OUR TURN'
), b AS (
    SELECT
        CASE WHEN last_contact_in_days <= 1  THEN 0
             WHEN last_contact_in_days <= 2  THEN 1
             WHEN last_contact_in_days <= 3  THEN 2
             WHEN last_contact_in_days <= 7  THEN 3
             WHEN last_contact_in_days <= 14 THEN 4
             WHEN last_contact_in_days <= 30 THEN 5
             ELSE 6 END AS ord,
        CASE WHEN last_contact_in_days <= 1  THEN '0-1 days (inside 24h SLA)'
             WHEN last_contact_in_days <= 2  THEN '2 days'
             WHEN last_contact_in_days <= 3  THEN '3 days'
             WHEN last_contact_in_days <= 7  THEN '4-7 days (major)'
             WHEN last_contact_in_days <= 14 THEN '8-14 days'
             WHEN last_contact_in_days <= 30 THEN '15-30 days'
             ELSE '> 30 days (dropped)' END AS waiting_since_bucket,
        health_band, final_health_score
    FROM waiting
)
SELECT
    waiting_since_bucket,
    health_band,
    COUNT(*) AS clients,
    ROUND(AVG(final_health_score), 0) AS avg_health_score
FROM b
GROUP BY ord, waiting_since_bucket, health_band
ORDER BY ord, health_band